# Multi-Source Sequence Fetch

![Multi-Source Sequence Fetch](https://proto-bio.github.io/proto-assets/images/tool/sequence_fetch/hero.png)

This notebook demonstrates `run_sequence_fetch`, a thin orchestrator that retrieves biological sequences and structures from multiple public databases in a single call. It chains NCBI, UniProt, and PDB lookups with molecule-type routing and cross-fetcher ID sharing, so a UniProt ID resolved during protein retrieval is automatically reused when looking up a PDB structure. The notebook covers fetching a protein sequence by name, combining multiple molecule types in one request, processing batch queries across several targets, and bypassing name resolution by providing known accession identifiers directly.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("sequence_fetch")
display_overview("sequence_fetch")
display_docs_section("sequence_fetch", "Background")

## Available tools

In [ ]:
display_available_tools("sequence_fetch")

### `run_sequence_fetch`

`run_sequence_fetch` accepts a list of `SequenceFetchRequest` objects, each specifying a target gene or protein name, an organism for disambiguation, and the desired molecule types. It searches UniProt, NCBI, and PDB in sequence, sharing resolved IDs across fetchers so each database is queried at most once per request. Results are returned as a `SequenceFetchOutput` with one `SequenceFetchResult` per request, containing the fetched sequences, structures, resolved IDs, and any per-type errors.

In [ ]:
from proto_tools import SequenceFetchInput, SequenceFetchConfig, SequenceFetchRequest, run_sequence_fetch

In [ ]:
# Display docs
display_api_reference("sequence_fetch", "input", "run_sequence_fetch")

# Fetch the lac repressor protein sequence from Escherichia coli
inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="lacI",
            organism="Escherichia coli",
            sequence_types=["protein"],
        )
    ]
)

In [ ]:
# Display config docs
display_api_reference("sequence_fetch", "config", "run_sequence_fetch")

# Use default config | see docs above for all fields
config = SequenceFetchConfig()

In [ ]:
# Run the fetch
output = run_sequence_fetch(inputs, config)

In [ ]:
# Display output docs
display_api_reference("sequence_fetch", "output", "run_sequence_fetch")

# Inspect the fetched protein sequence
result = output.results[0]
print(f"Target: {result.target_name} ({result.organism})")
print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

seq = result.fetched_sequences[0]
print(f"Type: {seq.sequence_type}")
print(f"Source: {seq.source_database} ({seq.accession})")
print(f"Length: {seq.length}")
print(f"Preview: {seq.sequence[:60]}...")

#### Multi-Type Request

A single request can fetch multiple molecule types at once. Here we retrieve the protein sequence, coding DNA sequence, and an experimentally determined structure for TP53. The orchestrator resolves IDs once and shares them across fetchers, so a UniProt ID found during protein retrieval is reused when looking up the associated PDB structure.

In [ ]:
# Fetch protein, CDS, and structure for TP53 in a single request
multi_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="TP53",
            organism="Homo sapiens",
            sequence_types=["protein", "dna_cds", "structure"],
            refseq_accession="NM_000546.6",  # pin canonical TP53 mRNA for the correct CDS
        )
    ]
)

multi_output = run_sequence_fetch(multi_inputs, config)
result = multi_output.results[0]

print(f"Target: {result.target_name} ({result.organism})")
print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

for seq in result.fetched_sequences:
    preview = seq.sequence[:50] + ("..." if len(seq.sequence) > 50 else "")
    print(f"  {seq.sequence_type}: {seq.source_database} ({seq.accession}), length={seq.length}")
    print(f"    {preview}")

for struct in result.fetched_structures:
    res_str = f"{struct.resolution:.1f} A" if struct.resolution else "N/A"
    print(f"  structure: {struct.pdb_id}, {struct.method}, {res_str}")
    print(f"    {struct.title}")

#### Batch Requests

Multiple targets can be processed in a single call by including multiple `SequenceFetchRequest` objects. Each request is resolved independently against the upstream databases.

In [ ]:
# Fetch three proteins in one call
batch_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="lacI",
            organism="Escherichia coli",
            sequence_types=["protein"],
        ),
        SequenceFetchRequest(
            target_name="insulin",
            organism="Homo sapiens",
            sequence_types=["protein"],
        ),
        SequenceFetchRequest(
            target_name="GFP",
            organism="Aequorea victoria",
            sequence_types=["protein"],
        ),
    ]
)

batch_output = run_sequence_fetch(batch_inputs, config)

print(f"Completed: {batch_output.num_completed}/{batch_output.num_requests}")
print()

for result in batch_output.results:
    if result.fetched_sequences:
        seq = result.fetched_sequences[0]
        print(f"{result.target_name}: {seq.source_database} ({seq.accession}), length={seq.length}")
    else:
        print(f"{result.target_name}: {result.status} -- {result.errors}")

#### Direct ID Override

When you already have a UniProt accession or PDB ID, you can provide it directly to skip the name search step. This is faster and avoids any ambiguity from name resolution.

In [ ]:
# Use known UniProt and PDB IDs to skip name resolution
direct_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="TP53",
            organism="Homo sapiens",
            sequence_types=["protein", "structure"],
            uniprot_id="P04637",
            pdb_id="1TSR",
        )
    ]
)

direct_output = run_sequence_fetch(direct_inputs, config)
result = direct_output.results[0]

print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

seq = result.fetched_sequences[0]
print(f"Protein: {seq.accession}, length={seq.length}")

struct = result.fetched_structures[0]
print(f"Structure: {struct.pdb_id}, {struct.method}")

## Export Results

Fetched sequences can be saved to FASTA or JSON format for downstream analysis. The FASTA export includes structured headers with target name, sequence type, source database, and accession for each record.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("sequence_fetch_results", export_path=output_dir, file_format="fasta")
print(f"Exported to {output_dir / 'sequence_fetch_results.fasta'}")

output.export("sequence_fetch_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'sequence_fetch_results.json'}")